# API Data Freshness Check

Most recent available timestamp for each training data source, with lag in hours.

In [1]:
import io, zipfile
import requests
import pandas as pd

NOW = pd.Timestamp.now()
print(f"Now: {NOW.strftime('%Y-%m-%d %H:%M')}")

Now: 2026-04-02 20:29


In [2]:
# Open-Meteo — ERA5 archive, typically ~5-day lag; hourly resolution
r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 40.7128, "longitude": -74.0060,
    "start_date": (NOW - pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
    "end_date":   NOW.strftime("%Y-%m-%d"),
    "hourly": "temperature_2m",
    "timezone": "America/New_York",
})
r.raise_for_status()
h = r.json()["hourly"]
temps = pd.Series(h["temperature_2m"], index=pd.to_datetime(h["time"]))
latest_openmeteo = temps.last_valid_index()
lag_h = (NOW - latest_openmeteo).total_seconds() / 3600
print(f"Open-Meteo:  {latest_openmeteo}  ({lag_h:.0f}h lag)")

Open-Meteo:  2026-04-02 23:00:00  (-3h lag)


In [3]:
# NYISO — 5-min interval data; walk back up to 3 months until a zip is available
latest_nyiso = None
for p in pd.period_range(end=NOW, periods=3, freq="M")[::-1]:
    r = requests.get(
        f"https://mis.nyiso.com/public/csv/pal/{p.year}{p.month:02d}01pal_csv.zip",
        timeout=15,
    )
    if r.ok:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            raw = pd.concat([pd.read_csv(z.open(n)) for n in z.namelist()])
        latest_nyiso = pd.to_datetime(raw["Time Stamp"]).max()
        lag_h = (NOW - latest_nyiso).total_seconds() / 3600
        print(f"NYISO:       {latest_nyiso}  ({lag_h:.0f}h lag)")
        break

NYISO:       2026-04-02 20:15:00  (0h lag)


In [4]:
# MTA daily ridership (city-wide total) — sayj-mze2
r = requests.get("https://data.ny.gov/resource/sayj-mze2.json",
                 params={"$order": "date DESC", "$limit": "1"}, timeout=30)
r.raise_for_status()
latest_mta_daily = pd.to_datetime(r.json()[0]["date"])
lag_h = (NOW - latest_mta_daily).total_seconds() / 3600
print(f"MTA:   {latest_mta_daily}  ({lag_h:.0f}h lag)")
# Note: MTA hourly-by-borough dataset (f462-ka72 / 5wq4-mkjj) was evaluated
# but has a ~180h publication lag — too stale for production use (8-day feature).

# NYC 311 — sub-hour timestamp precision
r = requests.get("https://data.cityofnewyork.us/resource/erm2-nwe9.json",
                 params={"$order": "created_date DESC", "$limit": "1"}, timeout=30)
r.raise_for_status()
latest_311 = pd.to_datetime(r.json()[0]["created_date"])
lag_h = (NOW - latest_311).total_seconds() / 3600
print(f"311:   {latest_311}  ({lag_h:.0f}h lag)")

MTA:   2026-04-01 00:00:00  (44h lag)
311:   2026-04-01 02:56:47  (42h lag)


In [5]:
# NYC Motor Vehicle Crashes — Socrata h9gi-nx95, server-side daily aggregation
r = requests.get("https://data.cityofnewyork.us/resource/h9gi-nx95.json", params={
    "$order": "crash_date DESC",
    "$limit": "1",
}, timeout=30)
r.raise_for_status()
latest_crashes = pd.to_datetime(r.json()[0]["crash_date"])
lag_h = (NOW - latest_crashes).total_seconds() / 3600
print(f"Crashes:     {latest_crashes}  ({lag_h:.0f}h lag)")

# Rate limit check — time a realistic aggregated query (one month of daily counts)
import time
t0 = time.perf_counter()
r2 = requests.get("https://data.cityofnewyork.us/resource/h9gi-nx95.json", params={
    "$select": "date_trunc_ymd(crash_date) as date, count(*) as crashes, "
               "sum(case(contributing_factor_vehicle_1='Pavement Slippery',1,true,0)) as slippery",
    "$group":  "date_trunc_ymd(crash_date)",
    "$where":  "crash_date >= '2024-12-01T00:00:00' AND crash_date <= '2024-12-31T23:59:59'",
    "$order":  "date ASC",
    "$limit":  "50",
}, timeout=30)
elapsed = time.perf_counter() - t0
r2.raise_for_status()
sample = pd.DataFrame(r2.json())
sample["date"] = pd.to_datetime(sample["date"]).dt.normalize()
print(f"  aggregated query: {len(sample)} days in {elapsed:.2f}s")
print(f"  sample: {sample.head(3).to_dict('records')}")

Crashes:     2026-03-30 00:00:00  (92h lag)
  aggregated query: 31 days in 0.36s
  sample: [{'date': Timestamp('2024-12-01 00:00:00'), 'crashes': '181', 'slippery': '1'}, {'date': Timestamp('2024-12-02 00:00:00'), 'crashes': '266', 'slippery': '1'}, {'date': Timestamp('2024-12-03 00:00:00'), 'crashes': '258', 'slippery': '0'}]


In [6]:
# DOT Traffic Speeds — real-time link speeds across NYC road network
r = requests.get("https://data.cityofnewyork.us/resource/i4gi-tjb9.json", params={
    "$order": "data_as_of DESC",
    "$limit": "500",
}, timeout=30)
r.raise_for_status()
df_dot = pd.DataFrame(r.json())
df_dot["speed"]      = pd.to_numeric(df_dot["speed"], errors="coerce")
df_dot["data_as_of"] = pd.to_datetime(df_dot["data_as_of"])
latest_dot = df_dot["data_as_of"].max()
lag_h = (NOW - latest_dot).total_seconds() / 3600
print(f"DOT speeds:  {latest_dot}  ({lag_h:.1f}h lag)")
print(f"  Links: {len(df_dot)} | Avg speed: {df_dot['speed'].mean():.1f} mph")
print(f"  By borough: {df_dot.groupby('borough')['speed'].mean().round(1).to_dict()}")

DOT speeds:  2026-04-02 18:48:13  (1.7h lag)
  Links: 500 | Avg speed: 28.1 mph
  By borough: {'Bronx': 27.1, 'Brooklyn': 22.9, 'Manhattan': 13.0, 'Queens': 36.9, 'Staten Island': 33.9}


In [7]:
summary = pd.DataFrame.from_dict({
    "Open-Meteo (ERA5)": latest_openmeteo,
    "NYISO Zone J":      latest_nyiso,
    "MTA Ridership":     latest_mta_daily,
    "NYC 311":           latest_311,
    "Crashes":           latest_crashes,
    "DOT Traffic Speeds": latest_dot,
}, orient="index", columns=["latest_timestamp"])
summary["lag_hours"] = summary["latest_timestamp"].apply(
    lambda t: round((NOW - t).total_seconds() / 3600, 1)
)
summary.index.name = "source"
summary

,latest_timestamp,lag_hours
source,,
Open-Meteo (ERA5),2026-04-02 23:00:00,-2.5
NYISO Zone J,2026-04-02 20:15:00,0.2
MTA Ridership,2026-04-01 00:00:00,44.5
NYC 311,2026-04-01 02:56:47,41.5
Crashes,2026-03-30 00:00:00,92.5
DOT Traffic Speeds,2026-04-02 18:48:13,1.7
